# CSA Phase 5c v2 — RAGOrigin Mode B (memory-safe)

**Author:** Monirul I. Mahmud | Supervisor: Dr. Justin Zhan

Fixes the v1 crash. v1's perturbation operators exploded document length (duplicating a
doc many times, zero-width char between every character), which overflowed the 12 GB GPU
on RAGOrigin's long documents and caused a CUDA OOM at case 40.

**v2 fixes (all uniform across cases):**
- Every perturbed document is truncated to MAX_DOC_TOKENS before scoring.
- Duplication is at most 2x; the distractor is capped; the zero-width operator is removed.
- `torch.cuda.empty_cache()` after every case; over-long or OOM candidates are skipped,
  not fatal.
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` set before torch loads, to reduce
  fragmentation.
- Resumes from any partial `ragorigin_modeb_results.json`.

Run: Restart & Run All. Test with N_TEST=3 first, then set N_TEST=None.


## 1. Config (sets the CUDA allocator flag before importing torch)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
import sys, json, time, math, random, warnings, gc
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import get_token

PROJECT_DIR   = r"C:\Users\mahmu\CSA_Project"
RAGORIGIN_DIR = os.path.join(PROJECT_DIR, "RAG-Responsibility-Attribution")
FEEDBACK_JSON = os.path.join(RAGORIGIN_DIR, "attack_feedback", "PRAGB", "k5_m5_e5_gpt-4o-mini.json")
PROXY_MODEL   = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE="cuda:0"; STEALTH=0.50; SEED=2; NUM_Q=100; CLEAN_DISTRACT=15
MAX_DOC_TOKENS=400          # hard cap on any (perturbed) document length
N_RAGORIGIN=50
RECORDS_DIR=os.path.join(PROJECT_DIR,"records"); OUT=os.path.join(RECORDS_DIR,"phase5c"); os.makedirs(OUT,exist_ok=True)
HF_TOKEN=get_token(); assert HF_TOKEN
with open(FEEDBACK_JSON, encoding="utf-8") as f: raw=json.load(f)
print("Loaded PRAGB events:", len(raw))


## 2. Scoring, poison builder, cdist (Phase 1b) + length-capped perturbation

In [ ]:
def make_poison(q,ta,resp,s):
    if s<0.35: return f"{q}. {resp}"
    if s<0.70: return f"{resp} This figure of {ta} is widely documented."
    return (f"According to production records, the figure commonly cited is {ta}. "
            f"Coverage in trade publications reflected the same detail.")
MP1='Below is a query from a user and a relevant context. Answer the question given the information in the context. \n\n\nContext: [context] \n\nQuery: [question] \n\nAnswer:'
MP2='Below is a query from a user and a relevant context. Answer the question given the information in the context. \n\n\nContext: [context] \n\nQuery:'
def w1(c,q): return MP1.replace('[context]',c).replace('[question]',q)
def w2(c):   return MP2.replace('[context]',c)
def zn(d):
    d=np.array(d,float); s=d.std(); return np.zeros_like(d) if s==0 else (d-d.mean())/s

print("Loading proxy:", PROXY_MODEL)
tok=AutoTokenizer.from_pretrained(PROXY_MODEL, token=HF_TOKEN)
proxy=AutoModelForCausalLM.from_pretrained(PROXY_MODEL, dtype=torch.bfloat16,
      attn_implementation="eager", device_map=DEVICE, token=HF_TOKEN); proxy.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

def truncate_doc(text, max_tokens=MAX_DOC_TOKENS):
    ids=tok(text, add_special_tokens=False)["input_ids"]
    if len(ids)<=max_tokens: return text
    return tok.decode(ids[:max_tokens], skip_special_tokens=True)

def closs(ctx,resp):
    ii=tok(ctx+' '+resp,return_tensors="pt")["input_ids"].to(DEVICE); ci=tok(ctx,return_tensors="pt")["input_ids"]
    li=ii.clone(); li[:,:ci.shape[1]]=-100
    with torch.no_grad(): out=proxy(ii,labels=li).loss.item()
    del ii,ci,li; return out
def score_docs(ctx,q,resp,rs):
    ctx=[truncate_doc(c) for c in ctx]        # safety cap on every doc
    a=[closs(w1(c,q),resp) for c in ctx]; qq=[closs(w2(c),q) for c in ctx]
    return np.array([(x+y+z)/3 for x,y,z in zip(zn(-np.array(a)),zn(-np.array(qq)),zn(np.array(rs)))])
def two_means(x,iters=25):
    x=np.asarray(x,float); c=np.array([x.min(),x.max()]); an=np.zeros(len(x),int)
    for _ in range(iters):
        an=np.abs(x[:,None]-c[None,:]).argmin(1)
        for k in (0,1):
            if (an==k).any(): c[k]=x[an==k].mean()
    return float(abs(c[1]-c[0]))
def graded(sc,n_p):
    order=np.argsort(sc)[::-1]; return len(set(order[:n_p].tolist())&set(range(n_p)))/n_p
Q_HAT=float(pd.read_csv(os.path.join(RECORDS_DIR,"phase4","fitted_thresholds_alpha10.csv")).query("tool=='ragorigin'").iloc[0].q_hat)
print("RAGOrigin q_hat =", round(Q_HAT,4))


## 3. Rebuild attacked cases (same seed/loop as the frozen file) and baseline

In [ ]:
rng=np.random.default_rng(SEED); cases=[]
for item in raw[:NUM_Q]:
    q=item["question"]; ta=item["target_answer"]; resp=item["RAG_response"]
    clean=[(t,r) for t,l,r in zip(item['context_texts'],item['context_labels'],item['retrieval_scores']) if not l][:CLEAN_DISTRACT]
    if len(clean)<3: continue
    ct=[t for t,_ in clean]; cr=[r for _,r in clean]; n_p=sum(item['context_labels']); mid=(min(cr)+max(cr))/2
    prs=[mid+rng.uniform(-0.02,0.02) for _ in range(n_p)]
    ctx=[make_poison(q,ta,resp,STEALTH) for _ in range(n_p)]+ct; rs=prs+cr
    cases.append(dict(case_id=f"ragorigin_nq_{item['question_id']}",question=q,target=ta,resp=resp,ctx=ctx,rs=rs,n_poison=n_p))
for c in cases:
    sc=score_docs(c["ctx"],c["question"],c["resp"],c["rs"]); c["bs"]=sc; c["bc"]=two_means(sc); c["ok"]=graded(sc,c["n_poison"])>=0.5
    gc.collect(); torch.cuda.empty_cache()
print(f"Reconstructed accuracy = {np.mean([c['ok'] for c in cases]):.3f} (expect ~0.82)")


## 4. Select + memory-safe attack

In [ ]:
elig=[c for c in cases if c["ok"] and c["bc"]>=Q_HAT]; elig.sort(key=lambda c:c["bc"])
sel = elig if len(elig)<=N_RAGORIGIN else [elig[i] for i in sorted(set(np.linspace(0,len(elig)-1,N_RAGORIGIN).round().astype(int)))]
print(f"eligible={len(elig)} selected={len(sel)}")

def kws(q,k=8):
    stop=set("the a an of to in is are and or for on with what who when where which how why did was were does do this that these those it as at by from".split())
    ws=[w.strip("?.,'\"").lower() for w in q.split()]; ws=[w for w in ws if w and w not in stop and len(w)>2]
    return ws[:k] if ws else ["information"]

def rago_attack(case, max_q=20, patience=5):
    q,ta,resp,n_p=case["question"],case["target"],case["resp"],case["n_poison"]
    ctx=list(case["ctx"]); rs=list(case["rs"]); kw=kws(q); clean_idx=list(range(n_p,len(ctx)))
    dis=" "+" ".join(kw)+". "+" ".join(reversed(kw))+". "
    final=case["bs"]
    def best_rank(sc):
        order=list(np.argsort(sc)[::-1]); return min(order.index(i) for i in clean_idx)
    bestm=best_rank(final); used=0; stale=0; succ=False
    for i in range(max_q):
        used+=1; order=list(np.argsort(final)[::-1]); target=min(clean_idx,key=lambda j:order.index(j))
        reps=min(1+i//4, 3); op=i%3; cand=list(ctx)
        if op==0:   cand[target]=cand[target]+dis*reps
        elif op==1: cand[target]=cand[target]+" "+cand[target]              # duplicate once (2x max)
        else:       cand[target]=dis+cand[target]                          # prepend distractor
        cand[target]=truncate_doc(cand[target])                            # hard length cap
        try:
            sc=score_docs(cand,q,resp,rs)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache(); stale+=1
            if stale>=patience: break
            continue
        g=graded(sc,n_p); cd=two_means(sc)
        if g<0.5 and cd>=Q_HAT: ctx=cand; final=sc; succ=True; break
        m=best_rank(sc)
        if m<bestm: bestm=m; ctx=cand; final=sc; stale=0
        else:
            stale+=1
            if stale>=patience: break
    g=graded(final,n_p); cd=two_means(final)
    gc.collect(); torch.cuda.empty_cache()
    return {"case_id":case["case_id"],"status":"done","tool":"ragorigin","backend":"llama3.2-3b",
            "base_cdist":round(float(case["bc"]),5),"queries_used":used,"final_graded":round(float(g),3),
            "final_cdist":round(float(cd),5),"q_hat":round(Q_HAT,5),"modeB_success":bool(succ and g<0.5 and cd>=Q_HAT)}
print("attack ready")


## 5. Run (test first, then full) + Wilson CI summary

In [ ]:
N_TEST=3        # <-- None for the full run
todo=sel if N_TEST is None else sel[:N_TEST]
rp=os.path.join(OUT,"ragorigin_modeb_results.json")
done={r["case_id"]:r for r in json.load(open(rp,encoding="utf-8"))} if os.path.isfile(rp) else {}
done={k:v for k,v in done.items() if v.get("status")=="done"}
results=list(done.values())
for i,case in enumerate(todo,1):
    if case["case_id"] in done: print(f"[{i}/{len(todo)}] {case['case_id']} cached"); continue
    t0=time.time(); r=rago_attack(case); results.append(r)
    json.dump(results, open(rp,"w",encoding="utf-8"), indent=2)
    print(f"[{i}/{len(todo)}] {case['case_id']:20s} success={r['modeB_success']} graded={r['final_graded']} cdist={r['final_cdist']} ({time.time()-t0:.0f}s)", flush=True)

def wilson(k,n,z=1.96):
    if n==0: return (0,0,0)
    p=k/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=(z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/d
    return p,max(0,c-h),min(1,c+h)
dd=[r for r in results if r.get("status")=="done"]; n=len(dd); k=sum(r["modeB_success"] for r in dd); p,lo,hi=wilson(k,n)
print(f"\nRAGOrigin: n={n} successes={k} rate={p:.3f} 95% CI [{lo:.3f},{hi:.3f}]")
if n: pd.DataFrame(dd).to_csv(os.path.join(OUT,"ragorigin_modeb_summary.csv"),index=False)
